# IMPORTS

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# SPLIT DATA

**Load dữ liệu**

In [ ]:
# ── 1. Load dữ liệu
df_train = pd.read_csv('/content/mental_heath_unbanlanced.csv', encoding='utf-8')
df_test  = pd.read_csv('/content/mental_health_combined_test.csv', encoding='utf-8')

print('Train columns:', df_train.columns.tolist())
print('Test  columns:', df_test.columns.tolist())

Train columns: ['Unique_ID', 'text', 'status']
Test  columns: ['text', 'status']


**Kiểm tra leak**

In [ ]:
# ── Kiểm tra Data Leakage giữa df_train và df_test
train_texts = set(df_train['text'].unique())
test_texts = set(df_test['text'].unique())

# Tìm giao điểm (các text xuất hiện ở cả 2 tập)
leakage_texts = train_texts.intersection(test_texts)

print(f"--- KIỂM TRA DATA LEAKAGE ---")
print(f"Số lượng text duy nhất trong Train: {len(train_texts):,}")
print(f"Số lượng text duy nhất trong Test : {len(test_texts):,}")

if len(leakage_texts) > 0:
    print(f"\n⚠️ CẢNH BÁO: Phát hiện {len(leakage_texts)} văn bản bị trùng lặp (leak) giữa tập Train và Test!")
    print(f"Tỷ lệ leak so với tập Test: {len(leakage_texts)/len(test_texts)*100:.2f}%")

    # Hiển thị thử 5 mẫu bị leak
    print("\nVí dụ 5 mẫu bị leak:")
    for i, txt in enumerate(list(leakage_texts)[:5]):
        print(f"{i+1}. {txt[:100]}...")
else:
    print("\n✅ Tuyệt vời: Không phát hiện data leak (trùng lặp nội dung) giữa Train và Test.")
print("----------------------------")

--- KIỂM TRA DATA LEAKAGE ---
Số lượng text duy nhất trong Train: 48,945
Số lượng text duy nhất trong Test : 992

⚠️ CẢNH BÁO: Phát hiện 496 văn bản bị trùng lặp (leak) giữa tập Train và Test!
Tỷ lệ leak so với tập Test: 50.00%

Ví dụ 5 mẫu bị leak:
1. I want the aliens to get me out of here.So, life is shit for me, constant pain, no energy, meds won'...
2. It would be kinder to everyone to just end it, why can’t I do it?I know I would be better off dead. ...
3. How do I tell my husband?Firstly, I am not suicidal. I hope it's ok to post here still. 

I've strug...
4. Well crap I had a meme I was gonna post then I remembered it’s text post weekend so y’all will have ...
5. Does anyone else pin their video on Zoom cause you’re too self conscious and need to see yourself? S...
----------------------------


**Gộp data**

In [ ]:
# ── 2. Gộp 2 tập lại
df_all = pd.concat([df_train[['text', 'status']], df_test], ignore_index=True)
print(f'\nTổng sau khi gộp: {len(df_all):,} mẫu')
print(df_all['status'].value_counts())


Tổng sau khi gộp: 50,604 mẫu
status
Normal        18639
Depression    14754
Suicidal      11460
Anxiety        5751
Name: count, dtype: int64


**Kiểm tra trùng lặp và loại bỏ trùng lặp**

In [ ]:
# Kiểm tra trùng lặp và thiếu nhất quán nhãn giữa 'text' và 'status'
print("--- Kiểm tra trùng lặp nội dung 'text' ---")
duplicated_text_mask = df_all.duplicated(subset=['text'], keep=False)
duplicated_texts_df = df_all[duplicated_text_mask].sort_values(by='text')

if duplicated_texts_df.empty:
    print("Không tìm thấy nội dung 'text' trùng lặp.")
else:
    print(f"Tìm thấy {len(duplicated_texts_df)} hàng có nội dung 'text' trùng lặp. Tổng số 'text' duy nhất bị trùng: {duplicated_texts_df['text'].nunique()}")

    print("\n--- Kiểm tra thiếu nhất quán nhãn 'status' cho các 'text' trùng lặp ---")
    inconsistent_labels = []
    for text, group in duplicated_texts_df.groupby('text'):
        if group['status'].nunique() > 1:
            inconsistent_labels.append(group)

    if not inconsistent_labels:
        print("Không tìm thấy sự thiếu nhất quán trong nhãn 'status' đối với các nội dung 'text' trùng lặp.")
    else:
        print(f"Tìm thấy {len(inconsistent_labels)} nội dung 'text' có nhãn 'status' không nhất quán:")
        for group_df in inconsistent_labels:
            print(f"\nNội dung Text: '{group_df['text'].iloc[0]}'\nCác nhãn Status tương ứng:\n{group_df[['text', 'status']]}")

print("--------------------------------------------------")

--- Kiểm tra trùng lặp nội dung 'text' ---
Tìm thấy 2179 hàng có nội dung 'text' trùng lặp. Tổng số 'text' duy nhất bị trùng: 1016

--- Kiểm tra thiếu nhất quán nhãn 'status' cho các 'text' trùng lặp ---
Tìm thấy 10 nội dung 'text' có nhãn 'status' không nhất quán:

Nội dung Text: 'All this work, all this pressure that everyone puts on you to succeed. To go to a good college, get a good job, the normal things a lot of parents ask. All for what? I work my entire life and then what? Am I supposed to enjoy my ofttimes from studying or working being an ugly, socially awkward loser? Not able to talk to anyone, have friends; even when doing normally enjoyed things (video games, time off, etc.) all I can think about is how everyone else is probably enjoying their time with other people. Am I working for something or am I just working for the sake of working just because everyone tells me that is what I am supposed to do. Do people only tell you it gets better for the sake of having one more w

In [ ]:
initial_rows = len(df_all)

# 1. Loại bỏ các văn bản có nhãn 'status' không nhất quán
# Lấy danh sách các 'text' có nhãn không nhất quán từ bước kiểm tra trước
inconsistent_texts = []
for text, group in df_all.groupby('text'):
    if group['status'].nunique() > 1:
        inconsistent_texts.append(text)

if inconsistent_texts:
    df_all_cleaned = df_all[~df_all['text'].isin(inconsistent_texts)].copy()
    print(f"Đã loại bỏ {len(inconsistent_texts)} văn bản có nhãn không nhất quán.")
else:
    df_all_cleaned = df_all.copy()
    print("Không tìm thấy văn bản có nhãn không nhất quán.")

# 2. Loại bỏ các hàng trùng lặp hoàn toàn (cả 'text' và 'status')
df_all_cleaned.drop_duplicates(subset=['text', 'status'], inplace=True)
print(f"Đã loại bỏ trùng lặp hoàn toàn. Còn lại {len(df_all_cleaned)} hàng.")

removed_rows = initial_rows - len(df_all_cleaned)
print(f"\nTổng số hàng đã loại bỏ: {removed_rows}")
print(f"Kích thước df_all_cleaned sau khi làm sạch: {len(df_all_cleaned)} hàng")

df_all = df_all_cleaned # Cập nhật df_all với dữ liệu đã làm sạch

# Kiểm tra lại sau khi làm sạch
print("\n--- Kiểm tra lại trùng lặp nội dung 'text' trên df_all sau làm sạch ---")
duplicated_text_mask_cleaned = df_all.duplicated(subset=['text'], keep=False)
duplicated_texts_df_cleaned = df_all[duplicated_text_mask_cleaned].sort_values(by='text')

if duplicated_texts_df_cleaned.empty:
    print("Không tìm thấy nội dung 'text' trùng lặp.")
else:
    print(f"Tìm thấy {len(duplicated_texts_df_cleaned)} hàng có nội dung 'text' trùng lặp. Tổng số 'text' duy nhất bị trùng: {duplicated_texts_df_cleaned['text'].nunique()}")

    print("\n--- Kiểm tra lại thiếu nhất quán nhãn 'status' cho các 'text' trùng lặp sau làm sạch ---")
    inconsistent_labels_cleaned = []
    for text, group in duplicated_texts_df_cleaned.groupby('text'):
        if group['status'].nunique() > 1:
            inconsistent_labels_cleaned.append(group)

    if not inconsistent_labels_cleaned:
        print("Không tìm thấy sự thiếu nhất quán trong nhãn 'status' đối với các nội dung 'text' trùng lặp.")
    else:
        print(f"Tìm thấy {len(inconsistent_labels_cleaned)} nội dung 'text' có nhãn 'status' không nhất quán:")
        for group_df in inconsistent_labels_cleaned:
            print(f"\nNội dung Text: '{group_df['text'].iloc[0]}'\nCác nhãn Status tương ứng:\n{group_df[['text', 'status']]}")

print("--------------------------------------------------")

Đã loại bỏ 10 văn bản có nhãn không nhất quán.
Đã loại bỏ trùng lặp hoàn toàn. Còn lại 49431 hàng.

Tổng số hàng đã loại bỏ: 1173
Kích thước df_all_cleaned sau khi làm sạch: 49431 hàng

--- Kiểm tra lại trùng lặp nội dung 'text' trên df_all sau làm sạch ---
Không tìm thấy nội dung 'text' trùng lặp.
--------------------------------------------------


**Tách train/val/test**

In [ ]:
# ── 3. Tách X, y
X = df_all['text']
y = df_all['status']

# ── 4. Chia train / val / test (70 / 10 / 20)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.125,
    random_state=42,
    stratify=y_temp
)

# ── 5. Kiểm tra kết quả
print(f'\n✅ Train : {len(X_train):,} mẫu')
print(f'✅ Val   : {len(X_val):,} mẫu')
print(f'✅ Test  : {len(X_test):,} mẫu')

print('\n--- Phân phối nhãn Train ---')
print(y_train.value_counts(normalize=True).round(3))

print('\n--- Phân phối nhãn Val ---')
print(y_val.value_counts(normalize=True).round(3))

print('\n--- Phân phối nhãn Test ---')
print(y_test.value_counts(normalize=True).round(3))


✅ Train : 34,601 mẫu
✅ Val   : 4,943 mẫu
✅ Test  : 9,887 mẫu

--- Phân phối nhãn Train ---
status
Normal        0.367
Depression    0.293
Suicidal      0.227
Anxiety       0.113
Name: proportion, dtype: float64

--- Phân phối nhãn Val ---
status
Normal        0.367
Depression    0.294
Suicidal      0.227
Anxiety       0.113
Name: proportion, dtype: float64

--- Phân phối nhãn Test ---
status
Normal        0.367
Depression    0.294
Suicidal      0.227
Anxiety       0.113
Name: proportion, dtype: float64


In [ ]:
# ── 6. Lưu lại thành file CSV
pd.DataFrame({'text': X_train, 'label': y_train}).to_csv('train_split.csv', index=False)
pd.DataFrame({'text': X_val,   'label': y_val}).to_csv('val_split.csv',   index=False)
pd.DataFrame({'text': X_test,  'label': y_test}).to_csv('test_split.csv',  index=False)
print('\n💾 Đã lưu: train_split.csv | val_split.csv | test_split.csv')


💾 Đã lưu: train_split.csv | val_split.csv | test_split.csv
